# Importaciones

In [1]:
include("dependencies.jl")
include("helpers.jl")
include("wrappers.jl")
# Cargamos los datos preparados en el notebook anterior al instante
JLD2.@load "datos_procesados.jld2" df_trainval df_test X_trainval y_trainval folds_trainval
n_features = 561;

# Modelos básicos y selección de atributos (20%)

In [2]:
# Definición de diccionarios de configuración
dic_filtros = Dict(
    "Sin_Filtrado" => nothing,
    "ANOVA" => MyANOVAFilter(n_features=n_features),
    "Pearson" => MyPearsonFilter(n_features=n_features),
    "Spearman" => MySpearmanFilter(n_features=n_features),
    "Kendall" => MyKendallFilter(n_features=n_features),
    "MI" => MyMIFilter(n_features=n_features),
    "RFE" => MyRFEFilter(n_features=n_features)
)

dic_reducciones = Dict(
    "Sin reducción" => IdentityTransformer(),
    "PCA" => PCA(variance_ratio=0.95),
    "ICA" => ICA(outdim=2, maxiter=10000,tol=0.5),
    "LDA" => LDA(method=:whiten, outdim=5) 
)

dic_modelos = Dict(
    "NeuralNetwork_50" => NeuralNetworkClassifier(builder=MLJFlux.MLP(hidden=(50,))),
    "NeuralNetwork_100" => NeuralNetworkClassifier(builder=MLJFlux.MLP(hidden=(100,))),
    "NeuralNetwork_100_50" => NeuralNetworkClassifier(builder=MLJFlux.MLP(hidden=(100, 50))),
    
    "KNN_1" => KNNClassifier(K=1),
    "KNN_10" => KNNClassifier(K=10),
    "KNN_20" => KNNClassifier(K=20),

    "SVM_0.1" => ProbabilisticSVC(cost=0.1),
    "SVM_0.5" => ProbabilisticSVC(cost=0.5),
    "SVM_1.0" => ProbabilisticSVC(cost=1.0)
);

In [3]:
function run_experiment(dic_filtros, dic_reducciones, dic_modelos, output_file; 
                              X=X_trainval, y=y_trainval, folds=folds_trainval)
    
    # --- 1. PREPARACIÓN DE RESULTADOS ---
    if isfile(output_file)
        println(">>> Archivo de checkpoint encontrado: $output_file")
        results_df = CSV.read(output_file, DataFrame)
        # Creamos un set de identificadores únicos para saltar lo ya hecho
        combinaciones_hechas = Set([
            (string(r.Filter), string(r.Reduction), string(r.Model)) 
            for r in eachrow(results_df)
        ])
        println(">>> $(length(combinaciones_hechas)) experimentos completados previamente.")
    else
        println(">>> Iniciando experimento desde cero.")
        results_df = DataFrame(
            Filter = String[], Reduction = String[], Model = String[],
            Accuracy_List = String[], 
            Accuracy_Mean = Float64[], F1_Score = Float64[], B_Accuracy = Float64[]
        )
        combinaciones_hechas = Set{Tuple{String, String, String}}()
    end
    
    # --- CORRECCIÓN DE MÉTRICAS ---
    # Usamos balanced_accuracy en lugar de recall para evitar errores multiclase
    measures = [accuracy, multiclass_f1score, balanced_accuracy]

    # --- 2. BUCLE PRINCIPAL ---
    for filt_name in sort(collect(keys(dic_filtros)))
        filt_model = dic_filtros[filt_name]
        
        for red_name in sort(collect(keys(dic_reducciones)))
            red_model = dic_reducciones[red_name]
            
            for mod_name in sort(collect(keys(dic_modelos)))
                mod_model = dic_modelos[mod_name]
                
                # Si ya existe esta combinación, saltamos
                if (filt_name, red_name, mod_name) in combinaciones_hechas
                    continue 
                end

                println("\n>>> Evaluando: [$filt_name] + [$red_name] + [$mod_name]")
                
                # 1. Definimos el Pipeline
                pipe = PersonalizedPipeline(
                    scaler    = MyMinMaxScaler(), 
                    filter    = filt_model,      
                    reduction = red_model,       
                    clf       = mod_model        
                )
                
                try
                    # 2. CREAMOS LA MACHINE (Vital para que MLJ funcione)
                    mach = machine(pipe, X, y) 

                    # 3. EVALUAMOS
                    # Usamos acceleration=Serial() para evitar choques con MLJFlux (Redes Neuronales)
                    evaluation = evaluate!(
                        mach, 
                        resampling = folds, 
                        measures = measures, 
                        verbosity = 0,
                        acceleration = CPUThreads()
                    )
                    
                    # Extraemos resultados
                    acc_per_fold = evaluation.measurement[1]
                    f1_val       = evaluation.measurement[2]
                    b_acc_val    = evaluation.measurement[3]
                    
                    acc_mean = mean(acc_per_fold)
                    # Convertimos lista a string seguro para CSV (usando ; como separador)
                    acc_str = replace(string(acc_per_fold), "," => ";")
                    
                    println("    ✓ Acc: $(round(acc_mean, digits=4)) | F1: $(round(f1_val, digits=4))")
                    
                    # Guardamos
                    push!(results_df, (filt_name, red_name, mod_name, acc_str, acc_mean, f1_val, b_acc_val))
                    CSV.write(output_file, results_df)
                    
                catch e
                    println("!!! ERROR en $filt_name + $red_name + $mod_name:")
                    # Mostramos el error real para debug
                    showerror(stdout, e)
                    println("")
                    
                    # Guardamos el error en el CSV para no perder la fila
                    push!(results_df, (filt_name, red_name, mod_name, "ERROR", NaN, NaN, NaN))
                    CSV.write(output_file, results_df)
                end
                
                # Limpiamos memoria
                GC.gc() 
            end
        end
    end
    
    println("\n=== Experimento Finalizado ===")
    return results_df
end

run_experiment (generic function with 1 method)

In [ ]:
# Ejecutar y guardar
df_resultados_basicos = run_experiment(
    dic_filtros, 
    dic_reducciones, 
    dic_modelos, 
    "resultados_modelos_basicos.csv"
)

>>> Iniciando experimento desde cero.

>>> Evaluando: [ANOVA] + [ICA] + [KNN_1]
    ✓ Acc: 0.5922 | F1: 0.5903

>>> Evaluando: [ANOVA] + [ICA] + [KNN_10]
    ✓ Acc: 0.6361 | F1: 0.6331

>>> Evaluando: [ANOVA] + [ICA] + [KNN_20]
    ✓ Acc: 0.6502 | F1: 0.6458
